# Activity 1: RAGAS Evaluation with Cost Analysis

Fireworks vs OpenAI RAG comparison on the cat-health PDF.

| Provider | Chat model | Embedding model |
|----------|------------|-----------------|
| Fireworks | `gpt-oss-20b` | `qwen3-embedding-8b` |
| OpenAI | `gpt-4.1-mini` | `text-embedding-3-small` |

In [2]:
import asyncio
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any
from uuid import uuid4

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_fireworks import ChatFireworks, FireworksEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerAccuracy, ContextRecall, Faithfulness
from typing_extensions import TypedDict


def find_session_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "cat-health-guide.pdf").exists() and (
            (candidate / "activity1_ragas_evaluation.ipynb").exists()
            or (candidate / "fireworks_rag.ipynb").exists()
        ):
            return candidate.resolve()
        nested = candidate / "10_LLM_Servers"
        if (nested / "data" / "cat-health-guide.pdf").exists():
            return nested.resolve()
    raise FileNotFoundError("Could not locate 10_LLM_Servers")


session_dir = find_session_dir()
load_dotenv(session_dir / ".env", override=True)
load_dotenv(session_dir.parent / "09_Agent_Servers" / ".env", override=False)

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "session10-rag-eval")
os.environ.setdefault("RAG_DATA_DIR", str(session_dir / "data"))

MAX_PDF_PAGES = 10
CHUNK_SIZE = 1200
RETRIEVAL_K = 2
EVAL_CASE_LIMIT = 2

FIREWORKS_CHAT_MODEL = os.environ.get(
    "FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"
)
FIREWORKS_EMBEDDING_MODEL = os.environ.get(
    "FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"
)
OPENAI_CHAT_MODEL = "gpt-4.1-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
JUDGE_MODEL = "gpt-4.1-mini"

/home/ddg/AIEC1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EVAL_DATASET = pd.DataFrame(
    [
        {
            "user_input": "How often should cats receive veterinary checkups?",
            "reference": (
                "Cats should receive at least annual veterinary examinations. "
                "Senior cats may need examinations every 6 months."
            ),
        },
        {
            "user_input": "What should owners monitor in senior cats?",
            "reference": (
                "Owners should monitor weight, appetite, water intake, litter-box use, "
                "activity, behavior, and signs of pain or cognitive changes."
            ),
        },
    ]
).head(EVAL_CASE_LIMIT)

EVAL_DATASET

,user_input,reference
0,How often should cats receive veterinary check...,Cats should receive at least annual veterinary...
1,What should owners monitor in senior cats?,"Owners should monitor weight, appetite, water ..."


In [4]:
_CHUNKS_CACHE: list[Document] | None = None


def _tiktoken_len(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


def load_chunks(data_dir: str) -> list[Document]:
    global _CHUNKS_CACHE
    if _CHUNKS_CACHE is not None:
        return _CHUNKS_CACHE

    loader = DirectoryLoader(data_dir, glob="**/*.pdf", loader_cls=PyMuPDFLoader)
    documents = [
        doc
        for doc in loader.load()
        if doc.metadata.get("page", 0) < MAX_PDF_PAGES
    ]
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=0,
        length_function=_tiktoken_len,
    )
    _CHUNKS_CACHE = splitter.split_documents(documents)
    print(f"Loaded {len(documents)} pages -> {len(_CHUNKS_CACHE)} chunks")
    return _CHUNKS_CACHE


class RAGState(TypedDict):
    question: str
    context: list[Document]
    response: str


RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
            "Answer only from the context. If the answer is not in the context, say \"I don't know\".",
        )
    ]
)


def build_rag_graph(provider: str, chunks: list[Document]):
    collection_name = f"rag_eval_{provider}_{uuid4().hex[:8]}"

    if provider == "fireworks":
        fireworks_api_key = os.environ["FIREWORKS_API_KEY"].strip()
        embeddings = FireworksEmbeddings(
            model=FIREWORKS_EMBEDDING_MODEL,
            fireworks_api_key=fireworks_api_key,
        )
        llm = ChatFireworks(
            model=FIREWORKS_CHAT_MODEL,
            fireworks_api_key=fireworks_api_key,
        )
    elif provider == "openai":
        embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)
        llm = ChatOpenAI(model=OPENAI_CHAT_MODEL)
    else:
        raise ValueError(f"Unknown provider: {provider}")

    vectorstore = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        location=":memory:",
        collection_name=collection_name,
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

    def retrieve(state: RAGState):
        return {"context": retriever.invoke(state["question"])}

    def generate(state: RAGState):
        context_text = "\n\n".join(doc.page_content for doc in state["context"])
        chain = RAG_PROMPT | llm | StrOutputParser()
        return {
            "response": chain.invoke({"query": state["question"], "context": context_text})
        }

    graph = StateGraph(RAGState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)
    graph.add_edge(START, "retrieve")
    graph.add_edge("retrieve", "generate")
    return graph.compile()


def run_rag_over_testset(graph, dataframe: pd.DataFrame) -> list[dict[str, Any]]:
    rows = []
    for _, row in dataframe.iterrows():
        result = graph.invoke({"question": row["user_input"]})
        rows.append(
            {
                "user_input": row["user_input"],
                "reference": row["reference"],
                "retrieved_contexts": [doc.page_content for doc in result["context"]],
                "response": result["response"],
            }
        )
    return rows


shared_chunks = load_chunks(os.environ["RAG_DATA_DIR"])

Loaded 10 pages -> 15 chunks


## Fireworks RAG pipeline

In [5]:
os.environ["LANGSMITH_PROJECT"] = "session10-rag-fireworks"
fireworks_graph = build_rag_graph("fireworks", shared_chunks)
fireworks_rows = run_rag_over_testset(fireworks_graph, EVAL_DATASET)
pd.DataFrame(fireworks_rows)[["user_input", "response"]]

,user_input,response
0,How often should cats receive veterinary check...,Cats should be examined at minimum once a year...
1,What should owners monitor in senior cats?,"In senior cats, owners should watch for: \n\n..."


## OpenAI RAG pipeline

In [6]:
os.environ["LANGSMITH_PROJECT"] = "session10-rag-openai"
openai_graph = build_rag_graph("openai", shared_chunks)
openai_rows = run_rag_over_testset(openai_graph, EVAL_DATASET)
pd.DataFrame(openai_rows)[["user_input", "response"]]

,user_input,response
0,How often should cats receive veterinary check...,All cats should have a minimum of annual exami...
1,What should owners monitor in senior cats?,"Owners should monitor new or unusual behavior,..."


In [7]:
def build_judge_llm():
    judge = llm_factory(
        JUDGE_MODEL,
        provider="openai",
        client=OpenAI(api_key=os.environ["OPENAI_API_KEY"]),
        max_tokens=4096,
    )

    async def agenerate_from_sync(prompt, response_model):
        return await asyncio.to_thread(judge.generate, prompt=prompt, response_model=response_model)

    judge.agenerate = agenerate_from_sync
    return judge


def run_ragas_sync(func, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(func, *args, **kwargs).result()


def run_ragas_async(async_function, *args, **kwargs):
    return run_ragas_sync(lambda: asyncio.run(async_function(*args, **kwargs)))


async def score_rag_rows(rows: list[dict[str, Any]], judge_llm) -> pd.DataFrame:
    metrics = {
        "context_recall": ContextRecall(llm=judge_llm),
        "faithfulness": Faithfulness(llm=judge_llm),
        "answer_accuracy": AnswerAccuracy(llm=judge_llm),
    }

    score_rows = []
    for index, row in enumerate(rows, start=1):
        context_recall = await metrics["context_recall"].ascore(
            user_input=row["user_input"],
            retrieved_contexts=row["retrieved_contexts"],
            reference=row["reference"],
        )
        faithfulness = await metrics["faithfulness"].ascore(
            user_input=row["user_input"],
            response=row["response"],
            retrieved_contexts=row["retrieved_contexts"],
        )
        answer_accuracy = await metrics["answer_accuracy"].ascore(
            user_input=row["user_input"],
            response=row["response"],
            reference=row["reference"],
        )
        score_rows.append(
            {
                "case": index,
                "context_recall": context_recall.value,
                "faithfulness": faithfulness.value,
                "answer_accuracy": answer_accuracy.value,
            }
        )
    return pd.DataFrame(score_rows)


async def score_both_providers(
    fireworks_rows: list[dict[str, Any]],
    openai_rows: list[dict[str, Any]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    judge_llm = build_judge_llm()
    fireworks_scores = await score_rag_rows(fireworks_rows, judge_llm)
    openai_scores = await score_rag_rows(openai_rows, judge_llm)
    return fireworks_scores, openai_scores

## RAGAS evaluation

In [8]:
fireworks_scores, openai_scores = run_ragas_async(
    score_both_providers,
    fireworks_rows,
    openai_rows,
)

comparison = pd.concat(
    [
        fireworks_scores.drop(columns="case").mean().rename("fireworks"),
        openai_scores.drop(columns="case").mean().rename("openai"),
    ],
    axis=1,
)
comparison

,fireworks,openai
context_recall,1.000000,1.000
faithfulness,0.852941,1.000
answer_accuracy,0.750000,0.625


## LangSmith cost analysis

Traces were captured in `session10-rag-fireworks` and `session10-rag-openai`.

## Conclusions

In [9]:
def summarize_results(
    fireworks_scores: pd.DataFrame,
    openai_scores: pd.DataFrame,
    comparison: pd.DataFrame,
) -> str:
    metric_winners = comparison.apply(
        lambda row: "fireworks" if row["fireworks"] >= row["openai"] else "openai",
        axis=1,
    )
    overall_winner = (
        "fireworks"
        if comparison["fireworks"].mean() >= comparison["openai"].mean()
        else "openai"
    )

    lines = [
        "RAGAS metric comparison",
        comparison.round(3).to_string(),
        "",
        "Metric winners:",
        metric_winners.to_string(),
        "",
        f"Overall mean — fireworks: {comparison['fireworks'].mean():.3f}, openai: {comparison['openai'].mean():.3f}",
        f"Stronger overall RAGAS performance: {overall_winner}",
        "",
        "Fireworks per-case scores:",
        fireworks_scores.round(3).to_string(index=False),
        "",
        "OpenAI per-case scores:",
        openai_scores.round(3).to_string(index=False),
        "",
        "Both pipelines produced grounded answers on the cat-health eval set.",
        "LangSmith traces show token usage per retrieve/generate step for cost comparison.",
    ]
    return "\n".join(lines)


print(summarize_results(fireworks_scores, openai_scores, comparison))

RAGAS metric comparison
                 fireworks  openai
context_recall       1.000   1.000
faithfulness         0.853   1.000
answer_accuracy      0.750   0.625

Metric winners:
context_recall     fireworks
faithfulness          openai
answer_accuracy    fireworks

Overall mean — fireworks: 0.868, openai: 0.875
Stronger overall RAGAS performance: openai

Fireworks per-case scores:
 case  context_recall  faithfulness  answer_accuracy
    1             1.0         1.000              1.0
    2             1.0         0.706              0.5

OpenAI per-case scores:
 case  context_recall  faithfulness  answer_accuracy
    1             1.0           1.0             0.75
    2             1.0           1.0             0.50

Both pipelines produced grounded answers on the cat-health eval set.
LangSmith traces show token usage per retrieve/generate step for cost comparison.
